# Milestone 5 · Error analysis

This notebook examines which fakes got through and which real clips were flagged at the validation decision threshold. It uses full-test exports from `src.score`. F-03 leads once its export is present; F-01 and F-02 help check whether particular patterns recur.

Run from within the repository. For duration and speaker analysis, point `M5_MANIFEST` to the version 4 `manifest_m4.csv`, or place it at `data/m4/manifests/manifest_m4.csv`. Audio-dependent work stays in private Kaggle; this notebook includes no playable audio.

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/metrics.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Run this notebook from within the project checkout.')
sys.path.insert(0, str(REPO))
from src.metrics import eer

RUNS = ('F-01', 'F-02', 'F-03')
TEST_SPLITS = ('test_indomain', 'test_unseen_gen', 'test_phone', 'test_itw', 'test_asvspoof')
KEY = ['split', 'clip_id']
DETAILS = ['label', 'language', 'dataset', 'generator']
MANIFEST = Path(os.environ.get('M5_MANIFEST', REPO / 'data/m4/manifests/manifest_m4.csv'))
MIN_CLASS_COUNT = 30  # Flag small denominators; do not rank speakers by these rates.

In [ ]:
def check_same_clips(reference, other, name):
    columns = KEY + DETAILS
    joined = reference[columns].merge(
        other[columns], on=KEY, how='outer', validate='one_to_one', indicator=True,
        suffixes=('', '_other'),
    )
    if not joined['_merge'].eq('both').all():
        missing = joined['_merge'].value_counts().to_dict()
        raise ValueError(f'{name}: scored clip coverage differs: {missing}')
    for col in DETAILS:
        if not joined[col].eq(joined[f'{col}_other']).all():
            raise ValueError(f'{name}: {col} does not match for all clip IDs')


predictions, thresholds, summaries = {}, {}, {}
for run in RUNS:
    folder = REPO / 'results' / run
    path = folder / 'predictions_full.csv.gz'
    if run == 'F-03' and not path.exists() and not (folder / 'results_full.json').exists():
        print('F-03 full-test export pending; using F-01 for detailed examples.')
        continue
    frame = pd.read_csv(path, keep_default_na=False)
    if set(frame['split']) != {'valid', *TEST_SPLITS} or frame.duplicated(KEY).any():
        raise ValueError(f'{run}: expected unique clip keys across valid and all five test splits')
    if not frame['label'].isin(['real', 'fake']).all() or not frame['p_fake'].between(0, 1).all():
        raise ValueError(f'{run}: invalid labels or probabilities')
    if predictions:
        check_same_clips(predictions['F-01'], frame, run)
    valid = frame[frame['split'].eq('valid')]
    _, threshold = eer(valid['label'].eq('fake'), valid['p_fake'])
    summary = json.loads((folder / 'results_full.json').read_text())
    if summary['run_id'] != run or summary['clips'] != len(frame):
        raise ValueError(f'{run}: CSV and summary refer to different runs or clip counts')
    if not math.isclose(threshold, summary['threshold'], abs_tol=1e-6):
        raise ValueError(f'{run}: found a mismatch in the validation EER threshold')
    fake = frame['label'].eq('fake')
    flagged = frame['p_fake'].ge(threshold)
    predictions[run] = frame.assign(
        run=run, threshold=threshold, predicted_label=flagged.map({True: 'fake', False: 'real'}),
        is_fake=fake, is_real=~fake, false_negative=fake & ~flagged, false_positive=~fake & flagged,
    )
    thresholds[run], summaries[run] = threshold, summary

lead_run = 'F-03' if 'F-03' in predictions else 'F-01'
test = pd.concat(predictions.values(), ignore_index=True)
test = test[test['split'].isin(TEST_SPLITS)]
lead = test[test['run'].eq(lead_run)]
display(pd.DataFrame({'threshold_from_valid': thresholds}))

## Threshold transfer

`p_fake >= threshold` means fake. Use each run’s EER threshold from `valid` on all five test splits. Divide false negatives by the number of fake clips and false positives by the number of real clips. A missing class has no rate. The `small_*_group` flags mark groups with fewer than 30 clips of that class.

In [ ]:
def error_rates(frame, groups):
    rates = frame.groupby(groups, as_index=False, observed=True).agg(
        n_fake=('is_fake', 'sum'), n_real=('is_real', 'sum'),
        missed_fakes=('false_negative', 'sum'), flagged_real=('false_positive', 'sum'),
    )
    rates['missed_pct'] = 100 * rates['missed_fakes'] / rates['n_fake'].where(rates['n_fake'].gt(0))
    rates['flagged_pct'] = 100 * rates['flagged_real'] / rates['n_real'].where(rates['n_real'].gt(0))
    rates['small_fake_group'] = rates['n_fake'].between(1, MIN_CLASS_COUNT - 1)
    rates['small_real_group'] = rates['n_real'].between(1, MIN_CLASS_COUNT - 1)
    return rates


transfer = error_rates(test, ['run', 'split'])
transfer['threshold'] = transfer['run'].map(thresholds)
transfer['accuracy'] = 1 - (transfer['missed_fakes'] + transfer['flagged_real']) / (
    transfer['n_fake'] + transfer['n_real']
)
for row in transfer.itertuples(index=False):
    if not math.isclose(row.accuracy, summaries[row.run]['accuracy'][row.split], abs_tol=1e-5):
        raise ValueError(f'{row.run}/{row.split}: accuracy disagrees with the full-score summary')
display(transfer.round(5))

unseen = transfer[transfer['run'].eq('F-01') & transfer['split'].eq('test_unseen_gen')].iloc[0]
unseen_auc = summaries['F-01']['auc']['test_unseen_gen']
print(f'F-01 unseen generators: {unseen.accuracy:.1%} accuracy at its valid threshold; ROC AUC {unseen_auc:.3f}.')

## Where the model makes mistakes

Break down mistakes at the fixed validation threshold. Attacks and generators describe fake clips, so those tables show missed fakes; consult the split and language tables for false positives. These are operating-point results, not per-group EER comparisons.

In [ ]:
print(f'Detailed breakdowns: {lead_run}')
by_language = error_rates(lead, ['split', 'language'])
generated = lead[lead['is_fake'] & ~lead['split'].isin(['test_asvspoof', 'test_itw'])]
by_generator = error_rates(generated, ['split', 'generator', 'language'])
attacks = lead[lead['is_fake'] & lead['split'].eq('test_asvspoof')]
by_attack = error_rates(attacks, ['generator']).rename(columns={'generator': 'attack'})
miss_columns = ['n_fake', 'missed_fakes', 'missed_pct', 'small_fake_group']
display(by_language.round(2))
display(by_generator[['split', 'generator', 'language', *miss_columns]].round(2))
display(by_attack[['attack', *miss_columns]].round(2))

focus = test[test['split'].eq('test_unseen_gen') & test['is_fake'] & (
    test['generator'].eq('indicf5') | (test['generator'].eq('mms-tts') & test['language'].eq('te'))
)]
focus_rates = error_rates(focus, ['run', 'generator', 'language'])
display(focus_rates[['run', 'generator', 'language', *miss_columns]].round(2))

In [ ]:
if MANIFEST.is_file():
    manifest = pd.read_csv(
        MANIFEST, keep_default_na=False, dtype={'speaker_id': str},
        usecols=KEY + DETAILS + ['speaker_id', 'duration_s'],
    )
    manifest = manifest[manifest['split'].isin(['valid', *TEST_SPLITS])]
    check_same_clips(predictions[lead_run], manifest, 'version 4 manifest')
    annotated = lead.merge(manifest[KEY + ['speaker_id', 'duration_s']], on=KEY, validate='one_to_one')
    annotated['duration'] = pd.cut(
        annotated['duration_s'], [0, 2, 4, 8, float('inf')], right=False,
        labels=['<2 s', '2–<4 s', '4–<8 s', '8+ s'],
    )
    if annotated['duration'].isna().any():
        raise ValueError('Some scored test clips have no duration bin.')
    by_duration = error_rates(annotated, ['split', 'duration'])
    display(by_duration.round(2))

    itw = annotated[annotated['split'].eq('test_itw')].copy()
    print(f'In-the-Wild clips without a speaker ID: {itw["speaker_id"].eq("").sum():,}')
    itw = itw[itw['speaker_id'].ne('')]
    aliases = {speaker: f'S{i:02}' for i, speaker in enumerate(sorted(itw['speaker_id'].unique()), 1)}
    by_speaker = error_rates(itw.assign(speaker=itw['speaker_id'].map(aliases)), ['speaker'])
    display(by_speaker.round(2))
elif 'M5_MANIFEST' in os.environ:
    raise FileNotFoundError(MANIFEST)
else:
    print('Set M5_MANIFEST to the version 4 manifest to include duration and speaker breakdowns.')

## Clips to review

Pick a seeded success for each true label and split when available. For mistakes, take the highest-scoring real clip flagged as fake and the lowest-scoring fake clip called real within each split. Break score ties by clip ID. These candidates are for private listening and saliency, not evidence of an audio cue by themselves.

In [ ]:
case_columns = ['split', 'clip_id', 'language', 'generator', 'label', 'p_fake', 'threshold', 'predicted_label']
case_groups = ['split', 'label']
correct = lead[~(lead['false_positive'] | lead['false_negative'])].sort_values('clip_id')
successes = correct.groupby(case_groups).sample(n=1, random_state=42)
false_positives = lead[lead['false_positive']].sort_values(['p_fake', 'clip_id'], ascending=[False, True])
false_negatives = lead[lead['false_negative']].sort_values(['p_fake', 'clip_id'])
mistakes = pd.concat([false_positives, false_negatives]).groupby(case_groups).head(1)
display(successes[case_columns].sort_values(case_groups))
display(mistakes[case_columns].sort_values(case_groups))
print(f'Splits with no false positives: {sorted(set(TEST_SPLITS) - set(false_positives["split"]))}')
print(f'Splits with no false negatives: {sorted(set(TEST_SPLITS) - set(false_negatives["split"]))}')

## Paired onset check

Set `M5_ONSET_OUTPUT` to the folder with the private Kaggle onset exports. Each split must account for every intact F-03 clip, either with a trimmed score or a short-clip exclusion. Recompute the intact EER on the matched subset. The 95% interval resamples clips within each class, using the same draws for the intact and trimmed scores. A difference measures sensitivity to the cut, which also changes clip length and window placement.

In [ ]:
def paired_eer(intact, trimmed, labels, rng, resamples):
    real = np.flatnonzero(~labels)
    fake = np.flatnonzero(labels)
    intact_eer = eer(labels, intact)[0]
    trimmed_eer = eer(labels, trimmed)[0]
    draws = np.empty(resamples)
    for i in range(resamples):
        indices = np.concatenate([rng.choice(real, len(real)), rng.choice(fake, len(fake))])
        draws[i] = eer(labels[indices], trimmed[indices])[0] - eer(labels[indices], intact[indices])[0]
    low, high = np.quantile(draws, [0.025, 0.975])
    return 100 * np.array([intact_eer, trimmed_eer, trimmed_eer - intact_eer, low, high])


def onset_report(folder, intact_predictions, resamples=1000):
    rows = []
    rng = np.random.default_rng(42)
    for split in TEST_SPLITS:
        kept = pd.read_csv(folder / f'{split}_predictions.csv.gz', keep_default_na=False)
        if set(kept['split']) != {'valid', split}:
            raise ValueError(f'{split}: expected only validation and the test split')
        kept = kept[kept['split'].eq(split)].set_index('clip_id', verify_integrity=True)
        excluded = pd.read_csv(folder / f'{split}_excluded.csv', keep_default_na=False)
        if not excluded['split'].eq(split).all():
            raise ValueError(f'{split}: exclusions contain rows from another split')
        excluded = excluded.set_index('clip_id', verify_integrity=True)
        intact = intact_predictions[intact_predictions['split'].eq(split)].set_index('clip_id', verify_integrity=True)
        if len(kept.index.intersection(excluded.index)) or set(kept.index.union(excluded.index)) != set(intact.index):
            raise ValueError(f'{split}: trimmed scores and exclusions must partition the intact predictions')
        for part in (kept, excluded):
            if not part['label'].eq(intact.loc[part.index, 'label']).all():
                raise ValueError(f'{split}: label mismatch')
        if not kept['p_fake'].between(0, 1).all():
            raise ValueError(f'{split}: invalid trimmed probability')
        labels = kept['label'].eq('fake').to_numpy()
        a, b, delta, low, high = paired_eer(
            intact.loc[kept.index, 'p_fake'].to_numpy(), kept['p_fake'].to_numpy(), labels, rng, resamples,
        )
        rows.append(dict(split=split, pairs=len(kept), excluded_real=excluded['label'].eq('real').sum(),
                         excluded_fake=excluded['label'].eq('fake').sum(), intact_eer_pct=a,
                         trimmed_eer_pct=b, change_pp=delta, ci_low_pp=low, ci_high_pp=high))
    return pd.DataFrame(rows)


if 'M5_ONSET_OUTPUT' in os.environ:
    if 'F-03' not in predictions:
        raise ValueError('Onset scores require the intact F-03 full export.')
    onset = onset_report(Path(os.environ['M5_ONSET_OUTPUT']), predictions['F-03'])
    display(onset.round(3))
else:
    print('Set M5_ONSET_OUTPUT after the full onset job finishes.')